# GOV-01 V2 Multi-label — final E6 baseline and E7 protected-test report

This is the final, one-time test report for two candidates trained on the **same N-RDD expansion train/validation split**:

- **E6:** fresh standard MobileNetV2 baseline (the E2-style baseline rebuilt on the expansion data).
- **E7:** E6-derived focused MobileNetV2 candidate.

E7 is preselected as the deployment candidate from validation evidence. This notebook reports both locked models on the untouched expansion test as final evidence; it does not retrain, tune thresholds, or change the preselection after seeing test results.

Attach these private Kaggle datasets before running:

1. `GOV-01 V2 Multilabel N-RDD2024 Expansion`
2. A small private dataset containing the E6 and E7 output ZIPs.

Run all cells once, download the final ZIP, and do not rerun this notebook.

In [ ]:
# Cell 1 — fixed configuration and safe input discovery. No training or validation data is read.
from pathlib import Path
import csv
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

CONDITIONS = ('crack', 'pothole', 'repaired_road', 'manhole_cover', 'unpaved_road', 'road_marking', 'speed_bump')
EXPECTED_TEST_IMAGES = 3978
PRESELECTED_DEPLOYMENT_CANDIDATE = 'E7'
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/gov01_final_e6_e7_test')
OUTPUT_ROOT = WORK_ROOT / 'evidence'
SESSION_MARKER = WORK_ROOT / 'protected_test_evaluated_this_session.json'

# Fixed from validation-only calibration. Do not edit after the protected test is read.
LOCKED_MODELS = {
    'E6': {
        'model_filename': 'e6_n_rdd_mobilenetv2_best.keras', 'image_size': (224, 224),
        'thresholds': {'crack': 0.51, 'pothole': 0.35, 'repaired_road': 0.47, 'manhole_cover': 0.35, 'unpaved_road': 0.95, 'road_marking': 0.50, 'speed_bump': 0.58},
    },
    'E7': {
        'model_filename': 'e7_focused_best.keras', 'image_size': (320, 320),
        'thresholds': {'crack': 0.46, 'pothole': 0.46, 'repaired_road': 0.56, 'manhole_cover': 0.49, 'unpaved_road': 0.71, 'road_marking': 0.50, 'speed_bump': 0.50},
    },
}

if SESSION_MARKER.exists():
    raise RuntimeError('This Kaggle session already evaluated the protected test. Do not run it again.')

def exactly_one_file(filename):
    matches = [path for path in INPUT_ROOT.rglob(filename) if path.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one {filename!r} under /kaggle/input; found {matches}')
    return matches[0]

def find_test_root():
    labels = [path for path in INPUT_ROOT.rglob('labels.csv') if path.parent.name == 'test']
    if len(labels) != 1:
        raise FileNotFoundError(f'Expected one attached protected-test labels.csv; found {labels}')
    return labels[0].parent.parent

E6_MODEL_PATH = exactly_one_file(LOCKED_MODELS['E6']['model_filename'])
E7_MODEL_PATH = exactly_one_file(LOCKED_MODELS['E7']['model_filename'])
TEST_ROOT = find_test_root()
TEST_LABELS_PATH = TEST_ROOT / 'test' / 'labels.csv'

print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('Preselected deployment candidate:', PRESELECTED_DEPLOYMENT_CANDIDATE)
print('E6 locked model:', E6_MODEL_PATH)
print('E7 locked model:', E7_MODEL_PATH)
print('Protected test root:', TEST_ROOT)

In [ ]:
# Cell 2 — verify and read only the protected test split.
def read_protected_test():
    paths, labels, masks = [], [], []
    with TEST_LABELS_PATH.open(newline='', encoding='utf-8') as source:
        for row in csv.DictReader(source):
            image_path = TEST_ROOT / 'test' / Path(row['materialized_image'].replace(chr(92), '/'))
            if not image_path.is_file():
                raise FileNotFoundError(f'Missing protected-test image: {image_path}')
            label_row, mask_row = [], []
            for condition in CONDITIONS:
                known, value = int(row[f'{condition}_known']), row[f'{condition}_present']
                if known not in (0, 1) or (known == 0 and value != '') or (known == 1 and value not in ('0', '1')):
                    raise ValueError(f'Invalid protected-test label: {row["record_id"]} / {condition}')
                label_row.append(float(value) if known else 0.0)
                mask_row.append(float(known))
            paths.append(str(image_path)); labels.append(label_row); masks.append(mask_row)
    return np.asarray(paths), np.asarray(labels, dtype='float32'), np.asarray(masks, dtype='float32')

test_paths, test_labels, test_masks = read_protected_test()
if len(test_paths) != EXPECTED_TEST_IMAGES:
    raise RuntimeError(f'Protected test has {len(test_paths)} images; expected {EXPECTED_TEST_IMAGES}. Stop and inspect the attached dataset.')
print(f'Protected test verified: {len(test_paths)} images')
for index, condition in enumerate(CONDITIONS):
    known = int(test_masks[:, index].sum())
    positive = int((test_labels[:, index] * test_masks[:, index]).sum())
    print(f'{condition:16} known={known:5} positive={positive:5}')
print('Unknown labels remain excluded from scoring.')

In [ ]:
# Cell 3 — one-time fixed-threshold predictions and metrics. No training or calibration occurs here.
def make_images(paths, size, batch_size=16):
    def load_image(path):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        return tf.cast(tf.image.resize(image, size), tf.float32)
    return tf.data.Dataset.from_tensor_slices(paths).map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

def evaluate_locked_model(name, path, settings):
    model = tf.keras.models.load_model(path, compile=False)
    probabilities = model.predict(make_images(test_paths, settings['image_size']), verbose=1)
    if probabilities.shape != test_labels.shape:
        raise RuntimeError(f'{name} returned {probabilities.shape}; expected {test_labels.shape}')
    per_condition, matrices, f1_values = {}, {}, []
    correct, known_total = 0, 0
    for index, condition in enumerate(CONDITIONS):
        known = test_masks[:, index].astype(bool)
        actual = test_labels[known, index].astype(int)
        threshold = settings['thresholds'][condition]
        predicted = (probabilities[known, index] >= threshold).astype(int)
        precision, recall, f1, _ = precision_recall_fscore_support(actual, predicted, average='binary', zero_division=0)
        matrix = confusion_matrix(actual, predicted, labels=[0, 1])
        tn, fp, fn, tp = matrix.ravel()
        per_condition[condition] = {'threshold_locked_before_test': float(threshold), 'known_images': int(len(actual)), 'positive_images': int(actual.sum()), 'precision': float(precision), 'recall': float(recall), 'f1': float(f1), 'true_negative': int(tn), 'false_positive': int(fp), 'false_negative': int(fn), 'true_positive': int(tp)}
        matrices[condition] = matrix
        f1_values.append(float(f1)); correct += int((actual == predicted).sum()); known_total += int(len(actual))
    return {'model': name, 'model_file': path.name, 'image_size': list(settings['image_size']), 'training_or_tuning_performed': False, 'protected_test_images': int(len(test_paths)), 'masked_macro_f1': float(np.mean(f1_values)), 'masked_label_accuracy': float(correct / known_total), 'per_condition': per_condition}, matrices

if SESSION_MARKER.exists():
    raise RuntimeError('Protected test was already evaluated in this session. Do not rerun it.')
e6_result, e6_matrices = evaluate_locked_model('E6', E6_MODEL_PATH, LOCKED_MODELS['E6'])
e7_result, e7_matrices = evaluate_locked_model('E7', E7_MODEL_PATH, LOCKED_MODELS['E7'])
SESSION_MARKER.parent.mkdir(parents=True, exist_ok=True)
SESSION_MARKER.write_text(json.dumps({'completed': True, 'models_reported': ['E6', 'E7'], 'preselected_candidate': PRESELECTED_DEPLOYMENT_CANDIDATE}, indent=2), encoding='utf-8')
print('E6 masked macro F1:', f"{e6_result['masked_macro_f1']:.4f}")
print('E7 masked macro F1:', f"{e7_result['masked_macro_f1']:.4f}")
print('Session marker written. Do not rerun this cell.')

In [ ]:
# Cell 4 — save a small final evidence package.
if 'e6_result' not in globals() or 'e7_result' not in globals():
    raise RuntimeError('Run the one-time scoring cell before saving evidence.')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
report = {'experiment': 'GOV-01 V2 multi-label final E6 baseline versus E7 protected-test report', 'preselected_deployment_candidate_before_test': PRESELECTED_DEPLOYMENT_CANDIDATE, 'models_were_locked_from_validation': True, 'protected_test_reused_for_tuning': False, 'training_or_tuning_performed': False, 'models': {'E6': e6_result, 'E7': e7_result}}
(OUTPUT_ROOT / 'final_e6_vs_e7_protected_test_metrics.json').write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
rows = [{'model': result['model'], 'condition': condition, **values} for result in (e6_result, e7_result) for condition, values in result['per_condition'].items()]
pd.DataFrame(rows).to_csv(OUTPUT_ROOT / 'final_e6_vs_e7_per_condition.csv', index=False)
figure, axes = plt.subplots(2, len(CONDITIONS), figsize=(21, 6))
for row_index, (name, matrices) in enumerate((('E6', e6_matrices), ('E7', e7_matrices))):
    for column_index, condition in enumerate(CONDITIONS):
        axis, matrix = axes[row_index, column_index], matrices[condition]
        axis.imshow(matrix, cmap='Blues'); axis.set_title(f'{name}: {condition}', fontsize=9)
        axis.set_xticks([0, 1], ['no', 'yes']); axis.set_yticks([0, 1], ['no', 'yes'])
        if column_index == 0: axis.set_ylabel('actual')
        if row_index == 1: axis.set_xlabel('predicted')
        cut = matrix.max() / 2 if matrix.max() else 0
        for y in range(2):
            for x in range(2): axis.text(x, y, str(matrix[y, x]), ha='center', va='center', fontsize=9, color='white' if matrix[y, x] > cut else 'black')
figure.suptitle('Final protected-test confusion matrices — thresholds fixed before test', y=1.02)
figure.tight_layout(); figure.savefig(OUTPUT_ROOT / 'final_e6_vs_e7_confusion_matrices.png', dpi=160, bbox_inches='tight'); plt.show()
summary = pd.DataFrame([{'model': 'E6', 'masked_macro_f1': e6_result['masked_macro_f1'], 'masked_label_accuracy': e6_result['masked_label_accuracy']}, {'model': 'E7', 'masked_macro_f1': e7_result['masked_macro_f1'], 'masked_label_accuracy': e7_result['masked_label_accuracy']}])
print(summary.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
print('Evidence folder:', OUTPUT_ROOT)

In [ ]:
# Cell 5 — package the final evidence for browser download. Keep it outside Git.
required = [OUTPUT_ROOT / 'final_e6_vs_e7_protected_test_metrics.json', OUTPUT_ROOT / 'final_e6_vs_e7_per_condition.csv', OUTPUT_ROOT / 'final_e6_vs_e7_confusion_matrices.png']
missing = [str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError(f'Missing final evidence: {missing}')
result_zip = Path('/kaggle/working/v2_multilabel_final_e6_vs_e7_protected_test_output.zip')
if result_zip.exists(): result_zip.unlink()
shutil.make_archive(str(result_zip.with_suffix('')), 'zip', OUTPUT_ROOT)
print('Completed final evidence ZIP:', result_zip)
print('Download it from Kaggle Output. Do not rerun the scoring cell or use the result for further tuning.')